In [ ]:
%pip install -q faiss-gpu

%pip install -q -U sentence_transformers rank_bm25 PyStemmer

!nvidia-smi

In [ ]:
import os

os.environ["HF_TOKEN"] = input()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")

dataset["train"]

In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import Dataset

retriever = SentenceTransformer("pameydorke/arcanumchat-retriever-bge-m3-v2")

train_split = dataset["train"]
val_split = dataset["validation"]

corpus_texts = list(dict.fromkeys(train_split["text"])) + list(dict.fromkeys(val_split["text"]))
text_to_idx = {t: i for i, t in enumerate(corpus_texts)}

In [ ]:
import Stemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_stemmer = Stemmer.Stemmer("english")

def tokenize(text: str):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    return _stemmer.stemWords(tokens)

tokenized_corpus = [tokenize(doc) for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)



In [ ]:
corpus_embeddings = retriever.encode(
    corpus_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

In [ ]:
def mine_hybrid_hard_negatives_hs(
    dataset,
    retriever,
    bm25_retriever,
    corpus_texts,
    corpus_embeddings,
    text_to_idx,
    stemmer,
    num_negatives=4,
    k_dense=100,
    k_bm25=100,
    rrf_k=60,
    hard_cutoff=10,
    medium_cutoff=30,
    hard_frac=0.5,
    medium_frac=0.3,
    query_batch_size=128,
    sim_batch_size=256,
    seed=42,
):
    """Mine reranker training negatives from a dense ∪ BM25 RRF pool.

    Returns a dataset with columns (anchor, positive, negative) — one row
    per (query, positive, negative) triplet, matching the `labeled-pair`
    format that CrossEncoder training expects.
    """
    queries   = dataset["anchor"]
    positives = dataset["positive"]

    # --- Encode all queries once ---
    query_embeddings = retriever.encode(
        queries,
        batch_size=query_batch_size,
        show_progress_bar=True,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    # --- Dense top-k in batches (avoids an [n_queries, n_corpus] blowup) ---
    dense_topk = []
    for start in range(0, len(query_embeddings), sim_batch_size):
        batch = query_embeddings[start:start + sim_batch_size]
        sim = util.cos_sim(batch, corpus_embeddings)
        topk = torch.argsort(sim, dim=1, descending=True)[:, :k_dense]
        dense_topk.append(topk.cpu().numpy())
    dense_topk = np.vstack(dense_topk)                 # [n_queries, k_dense]

    # --- Per-query: dense ∪ BM25 → RRF → sample negatives ---
    rows = []
    for i, (query, positive) in enumerate(tqdm(
        zip(queries, positives), total=len(queries), desc="Mining hybrid"
    )):
        positive_idx = text_to_idx.get(positive)

        dense_ranking = [{"id": int(idx)} for idx in dense_topk[i]]

        scores = bm25.get_scores(tokenize(query))
        bm25_topk = np.argsort(scores)[::-1][:k_bm25]
        bm25_ranking = [{"id": int(idx)} for idx in bm25_topk]

        fused = fuse(
            [dense_ranking, bm25_ranking],
            method="rrf",
            k=rrf_k,
            top_k=None,
        )
        fused_ids = [r.document_id for r in fused if r.document_id != positive_idx]

        # Difficulty bins from the fused pool
        hard_pool   = fused_ids[:hard_cutoff]
        medium_pool = fused_ids[hard_cutoff:medium_cutoff]
        easy_pool   = fused_ids[medium_cutoff:]

        n_hard   = max(0, round(num_negatives * hard_frac))
        n_medium = max(0, round(num_negatives * medium_frac))
        n_easy   = num_negatives - n_hard - n_medium

        rng = np.random.default_rng(seed + i)

        def sample(pool, n):
            if not pool or n <= 0:
                return []
            k = min(n, len(pool))
            return rng.choice(pool, size=k, replace=False).tolist()

        neg_ids  = sample(hard_pool,   n_hard)
        neg_ids += sample(medium_pool, n_medium)
        neg_ids += sample(easy_pool,   n_easy)

        # Fallback: random corpus docs if fused pool was too small
        while len(neg_ids) < num_negatives:
            j = int(rng.integers(0, len(corpus_texts)))
            if j != positive_idx and j not in neg_ids:
                neg_ids.append(j)

        for idx in neg_ids[:num_negatives]:
            rows.append({
                "anchor":   query,
                "positive": positive,
                "negative": corpus_texts[int(idx)],
            })

    return Dataset.from_list(rows)

In [ ]:
train_dataset = dataset["train"].select_columns(["query", "text"]).rename_columns({
    "query": "anchor", "text": "positive",
})
val_dataset = dataset["validation"].select_columns(["query", "text"]).rename_columns({
    "query": "anchor", "text": "positive",
})

hard_train_dataset = mine_hybrid_hard_negatives(
    dataset=train_dataset,
    retriever=retriever,
    bm25_retriever=bm25_retriever,
    corpus_texts=corpus_texts,
    corpus_embeddings=corpus_embeddings,
    text_to_idx=text_to_idx,
    stemmer=stemmer,
    num_negatives=4,
)

hard_val_dataset = mine_hybrid_hard_negatives(
    dataset=val_dataset,
    retriever=retriever,
    bm25_retriever=bm25_retriever,
    corpus_texts=corpus_texts,
    corpus_embeddings=corpus_embeddings,
    text_to_idx=text_to_idx,
    stemmer=stemmer,
    num_negatives=4,
)

In [ ]:
from collections import defaultdict

grouped = defaultdict(lambda: {"positive": [], "negative": []})

for anchor, document, label in zip(
    hard_val_dataset["anchor"], hard_val_dataset["positive"], hard_val_dataset["label"]
):
    key = "positive" if int(label) == 1 else "negative"
    grouped[anchor][key].append(document)

# Only keep queries that have at least one positive AND one negative
hard_eval_samples = [
    {"query": q, **docs}
    for q, docs in grouped.items()
    if docs["positive"] and docs["negative"]
]

print(f"Evaluation queries: {len(hard_eval_samples)}")

In [ ]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

evaluator = CrossEncoderRerankingEvaluator(
    samples=hard_eval_samples,
    at_k=20,
    name="val-hard-negatives",
    batch_size=64,
    show_progress_bar=True,
    always_rerank_positives=True,
)

In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1)

In [ ]:
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from datasets import Dataset

train_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in hard_train_dataset],
    "document": [p["positive"] for p in hard_train_dataset],
    "label": [p["label"] for p in hard_train_dataset]
})

val_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in hard_val_dataset],
    "document": [p["positive"] for p in hard_val_dataset],
    "label": [p["label"] for p in hard_val_dataset]
})

train_loss = BinaryCrossEntropyLoss(model)

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from transformers import EarlyStoppingCallback

args = CrossEncoderTrainingArguments(
    warmup_steps=0.1,
    lr_scheduler_type="linear",

    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps = 2,

    logging_steps=100,

    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_val-hard-negatives_ndcg@20",
    greater_is_better=True,

    eval_on_start=True,
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,

    evaluator=evaluator,
    loss=train_loss,
)

In [ ]:
trainer.train()

In [ ]:
model.push_to_hub("pameydorke/arcanumchat-reranker-bge-reranker-v2-m3-hybrid-search")